# CNU Campus ChatBot — 제출물 실행 안내

이 노트북 하나로 **Task 1 (질문 분류)**를 수행합니다. **Task 2·3 (챗봇 / 실시간)**은 아래 터미널 안내를 따르세요.

---

## Task 1 — 질문 분류 (이 노트북)

상단 메뉴 **런타임 → 모두 실행 (Run all)** 을 누르면 됩니다.

- 동봉된 `model/classifier.joblib` 로 `data/test_cls.json` 을 분류해 `outputs/cls_output.json` 을 만듭니다.
- 코랩에서 프로젝트 폴더를 자동으로 찾습니다. 못 찾으면 아래 코드 셀의 `PROJECT_RELATIVE_TO_DRIVE` 에
  드라이브(MyDrive) 기준 상대경로(예: `NLP_TermProject/Termproject_신해솔`)를 입력하고 다시 실행하세요.
- 외부 의존성은 `scikit-learn`, `joblib` 둘 뿐이며 자동 설치됩니다. (인터넷/클론 불필요)

## Task 2·3 — 챗봇 / 실시간 (코랩 터미널)

1. 코랩에서 **터미널**을 엽니다.
2. 프로젝트 폴더로 이동합니다:
   ```bash
   cd /content/drive/MyDrive/NLP_TermProject/Termproject_신해솔
   ```
3. 실행합니다:
   ```bash
   bash chatbot.sh
   ```
   - 그냥 **Enter** (또는 30초 대기) → 채점 출력파일 생성:
     `outputs/chat_output.json`, `outputs/realtime_output.json`
   - **`u`** 입력 후 Enter → 데모용 웹 UI 실행 (공개 URL 출력)
   - 최초 실행 시 의존성 설치 + 모델(약 5.5GB) 다운로드로 수 분 소요됩니다.

## 출력물

| 파일 | 태스크 |
| --- | --- |
| `outputs/cls_output.json` | Task 1 (이 노트북) |
| `outputs/chat_output.json` | Task 2 |
| `outputs/realtime_output.json` | Task 3 |


In [ ]:
# --- Task 1 설정: 의존성 고정 + 프로젝트 폴더 탐색 (self-contained) ---
import json
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

# 자동 탐색 실패 시: MyDrive 기준 프로젝트 폴더 상대경로를 입력하세요.
# 예) "NLP_TermProject/Termproject_신해솔"
PROJECT_RELATIVE_TO_DRIVE = ""

# classifier.joblib 가 학습된 환경과 동일하게 고정 (pickle 호환성 보장)
PINNED = {"scikit-learn": "1.7.2", "joblib": "1.5.3"}


def _installed(pkg: str) -> str | None:
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


def ensure_pinned_deps() -> None:
    """sklearn/joblib 를 import 하기 전에 메타데이터만으로 버전을 확인하고 필요 시 설치."""
    needed = [f"{p}=={v}" for p, v in PINNED.items() if _installed(p) != v]
    if not needed:
        return
    print("Installing pinned deps:", needed)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *needed])


def _is_project_root(path: Path) -> bool:
    return (path / "chatbot.sh").is_file() and (path / "model" / "classifier.joblib").is_file()


def _mount_drive_if_colab() -> Path | None:
    if not Path("/content").exists():
        return None
    my_drive = Path("/content/drive/MyDrive")
    if not my_drive.exists():
        try:
            from google.colab import drive  # type: ignore

            drive.mount("/content/drive")
        except Exception as exc:  # noqa: BLE001
            print("Google Drive mount skipped:", exc)
            return None
    return my_drive if my_drive.exists() else None


def find_project_root() -> Path:
    # 1) 명시적 환경변수
    env_root = os.environ.get("NLP_TERM_PROJECT_ROOT")
    if env_root and _is_project_root(Path(env_root).expanduser()):
        return Path(env_root).expanduser().resolve()

    # 2) 현재 경로에서 상위로 탐색
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if _is_project_root(candidate):
            return candidate

    # 3) 코랩 Google Drive 탐색
    my_drive = _mount_drive_if_colab()
    if my_drive is not None:
        if PROJECT_RELATIVE_TO_DRIVE:
            manual = (my_drive / PROJECT_RELATIVE_TO_DRIVE).resolve()
            if _is_project_root(manual):
                return manual
        patterns = ["Termproject*", "*/Termproject*", "*/*/Termproject*", "nlp-term", "*/nlp-term"]
        matches: list[Path] = []
        for pattern in patterns:
            for candidate in my_drive.glob(pattern):
                resolved = candidate.resolve()
                if _is_project_root(resolved) and resolved not in matches:
                    matches.append(resolved)
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            listing = "\n".join(f"  - {match}" for match in matches)
            raise RuntimeError(
                "여러 프로젝트 폴더가 발견되어 자동 선택을 중단했습니다. 이 셀 상단의 "
                "PROJECT_RELATIVE_TO_DRIVE 로 하나를 지정하세요:\n" + listing
            )

    raise RuntimeError(
        "프로젝트 폴더를 찾지 못했습니다. 이 셀 상단의 PROJECT_RELATIVE_TO_DRIVE 를 "
        "MyDrive 기준 상대경로(예: 'NLP_TermProject/Termproject_신해솔')로 설정한 뒤 다시 실행하세요."
    )


ensure_pinned_deps()
PROJECT_ROOT = find_project_root()
print("project root :", PROJECT_ROOT)


In [ ]:
# --- Task 1 실행: 분류 후 outputs/cls_output.json 생성 ---
import joblib

# 입력/출력은 항상 프로젝트 폴더 기준 (chatbot.sh 의 outputs/ 와 동일 위치)
input_path = PROJECT_ROOT / "data" / "test_cls.json"
output_path = PROJECT_ROOT / "outputs" / "cls_output.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

model = joblib.load(PROJECT_ROOT / "model" / "classifier.joblib")
rows = json.loads(input_path.read_text(encoding="utf-8"))
questions = [row["question"] for row in rows]
predictions = model.predict(questions)
labels = predictions.tolist() if hasattr(predictions, "tolist") else list(predictions)

result = [{"question": q, "label": int(label)} for q, label in zip(questions, labels)]
output_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"input  : {input_path}")
print(f"output : {output_path}")
print(f"rows   : {len(result)}")
for item in result[:3]:
    print("  ", item)
